In [ ]:
import json, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import make_scorer, f1_score, precision_score, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
df=pd.read_csv('../data/heart.csv')
y=df['target']
x=df.drop(columns=['target'])
num=x.select_dtypes(include=[int,float]).columns.tolist()
cat=[c for c in x.columns if c not in num]
pre=ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler())]),num),('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('oh',OneHotEncoder(handle_unknown='ignore'))]),cat)])
models={'logreg':LogisticRegression(max_iter=1000,class_weight='balanced'),'dt':DecisionTreeClassifier(class_weight='balanced',random_state=42),'rf':RandomForestClassifier(class_weight='balanced',random_state=42),'svm':SVC(kernel='rbf',probability=True,class_weight='balanced',random_state=42)}
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
scoring={'roc_auc':'roc_auc','f1':make_scorer(f1_score,zero_division=0),'precision':make_scorer(precision_score,zero_division=0),'recall':make_scorer(recall_score,zero_division=0),'accuracy':'accuracy'}
rows=[]
for name,est in models.items():
    pipe=Pipeline([('pre',pre),('clf',est)])
    s=cross_validate(pipe,x,y,cv=cv,scoring=scoring)
    rows.append({'model':name,'roc_auc':float(np.mean(s['test_roc_auc'])),'f1':float(np.mean(s['test_f1'])),'precision':float(np.mean(s['test_precision'])),'recall':float(np.mean(s['test_recall'])),'accuracy':float(np.mean(s['test_accuracy']))})
pd.DataFrame(rows).to_csv('../results/supervised_metrics.csv',index=False)
print(pd.read_csv('../results/supervised_metrics.csv'))